Notebook to train and test the GNN's

In [67]:
from gnn.ECC_GNN_V2 import MCC_GNN
from torch.utils.data import random_split
from torch_geometric.loader import DataLoader
import torch
import torch.nn as nn


Loading of datasets

In [73]:


# dataset = torch.load('../version1/graphswitharea.pt', weights_only=False)
# dataset = torch.load('../version1/graphssmaller.pt', weights_only=False)
# dataset = torch.load('../datasets/version1/morediversetesting2000.pt', weights_only=False)
# dataset = torch.load('../data generation/datasets/version1/montecarlotargets20.pt', weights_only=False)
# dataset = torch.load('../data generation/datasets/version1/radiationsmallnodessamestats.pt', weights_only=False)

#Gravity dataset
dataset = torch.load('../data generation/datasets/version1/morediversetestinggravity2000.pt', weights_only=False)

#Radiation dataset
#dataset = torch.load('../data generation/datasets/version1/morediversetestingbigger1000.pt', weights_only=False)

total_graphs = len(dataset)

#train test split
train_size = int(0.8 * total_graphs)
test_size = total_graphs - train_size

print(f"Total Graphs: {total_graphs} | Training on: {train_size} | Testing on: {test_size}")

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

Total Graphs: 2000 | Training on: 1600 | Testing on: 400


Get threshold statistics of loaded dataset

In [74]:

total_y = 0.0
total_samples = 0
global_min = 100
global_max = 0

for data in train_loader:
    total_y += data.y.sum().item()
    total_samples  += data.y.numel()

    batch_min = data.y.min().item()
    batch_max = data.y.max().item()


    if batch_min < global_min:
        global_min  = batch_min

    if  batch_max  > global_max:
        global_max  = batch_max

average_y = total_y / total_samples
print(f"average: {average_y:.4f}")
print (f"max: {global_max}, min: {global_min}")

average: 0.5883
max: 0.9778836369514465, min: 0.5000258684158325


save model training checkpoint

In [2]:
import torch

def save_checkpoint(model, optimizer, epoch, loss, filepath="gnn_checkpoint.pth"):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss
    }
    torch.save(checkpoint, filepath)
    print(f"Checkpoint saved at epoch {epoch} to {filepath}")

load trained model

In [3]:
def load_checkpoint(model, optimizer, filepath="gnn_checkpoint.pth"):
    """Loads the saved state back into the model and optimizer."""
    # Load the dictionary from the file
    checkpoint = torch.load(filepath, map_location='cpu')

    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    start_epoch = checkpoint['epoch']
    last_loss = checkpoint['loss']

    print(f"Resumed training from epoch {start_epoch} (Previous Loss: {last_loss:.4f})")

    return model, optimizer, start_epoch

In [6]:
model = MCC_GNN(edge_dimension=1, node_dimension= 4, hidden_dimension= 64)
#model = Advanced_Mobility_GNN(edge_dimension=1, node_dimension= 4, hidden_dimension= 64)



optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4) # Adam is the standard for GNNs
#criterion = nn.MSELoss() #MSE
# criterion = nn.L1Loss()
criterion = nn.SmoothL1Loss()
# model, optimizer, start_epoch = load_checkpoint(model, optimizer, "gnn_testing_radiation_wihtout_position_smoothingloss.pth")


In [103]:
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader


#model = Mobility_ECGNN(node_features_dim=3, edge_features_dim=1, hidden_dim=64)
model = MCC_GNN(edge_dimension=1, node_dimension= 4, hidden_dimension= 64)
#model = Advanced_Mobility_GNN(edge_dimension=1, node_dimension= 4, hidden_dimension= 64)

# model, optimizer, start_epoch = load_checkpoint(model, optimizer, "gnn_testing_radiation_50nodes.pth")

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4) # Adam is the standard for GNNs
#criterion = nn.MSELoss() #MSE
# criterion = nn.L1Loss()
criterion = nn.SmoothL1Loss()
# model, optimizer, start_epoch = load_checkpoint(model, optimizer, "gnn_testing_radiation_wihtout_position_smoothingloss.pth")

# training_graphs = torch.load('/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/testingpyg.pt', weights_only=False)
# train_loader = DataLoader(training_graphs, batch_size=32, shuffle=True)

#training loop
epochs = 100
FEATURE_TO_IGNORE = [2,3]

for epoch in range(epochs):
    total_loss = 0

    model.train()
    for batch_data in train_loader:

        batch_data.x[:, FEATURE_TO_IGNORE] = 0.0

        optimizer.zero_grad()
        predictions =  model(batch_data)

        loss = criterion(predictions.squeeze(), batch_data.y)
        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    total_test_error = 0

    with torch.no_grad():
     for batch_data in test_loader:
        batch_data.x[:, FEATURE_TO_IGNORE] = 0.0

        #prediciton
        predictions = model(batch_data)

        #loss
        loss = criterion(predictions.squeeze(), batch_data.y)
        total_test_error += loss.item()

#avg error
     avg_test_mse = total_test_error / len(test_loader)

#print progress and save
    if epoch % 1 == 0:
        print(f"Epoch {epoch} | Average Training Loss (MSE): {avg_loss:.6f}")
        print(f"Epoch {epoch} | Average Testing Loss (MSE): {avg_test_mse:.6f}")
        save_checkpoint(model, optimizer, epoch, loss,
                        "extra-trained model versions/gnn_testing_radiation_50nodessamestatts.pth")

Epoch 0 | Average Training Loss (MSE): 0.009197
Epoch 0 | Average Testing Loss (MSE): 0.003087
Checkpoint saved at epoch 0 to gnn_testing_radiation_50nodessamestatts.pth
Epoch 1 | Average Training Loss (MSE): 0.002845
Epoch 1 | Average Testing Loss (MSE): 0.002380
Checkpoint saved at epoch 1 to gnn_testing_radiation_50nodessamestatts.pth
Epoch 2 | Average Training Loss (MSE): 0.002639
Epoch 2 | Average Testing Loss (MSE): 0.002203
Checkpoint saved at epoch 2 to gnn_testing_radiation_50nodessamestatts.pth
Epoch 3 | Average Training Loss (MSE): 0.002355
Epoch 3 | Average Testing Loss (MSE): 0.002292
Checkpoint saved at epoch 3 to gnn_testing_radiation_50nodessamestatts.pth
Epoch 4 | Average Training Loss (MSE): 0.002420
Epoch 4 | Average Testing Loss (MSE): 0.002009
Checkpoint saved at epoch 4 to gnn_testing_radiation_50nodessamestatts.pth
Epoch 5 | Average Training Loss (MSE): 0.002280
Epoch 5 | Average Testing Loss (MSE): 0.001958
Checkpoint saved at epoch 5 to gnn_testing_radiation_50

load different model versions

In [27]:
from GAT_GNN_V2 import GAT_GNN

In [33]:
model = GAT_GNN(node_dimension=4, hidden_dimension=64)

# 3. Instantiate optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
criterion = nn.SmoothL1Loss()

model, optimizer, start_epoch = load_checkpoint(model, optimizer, "TGNN_radiation_wihtout_position_smoothingloss_gpu.pth")

In [55]:
model, optimizer, start_epoch = load_checkpoint(model, optimizer, "GAT_GNN_fully.pth")

RuntimeError: Error(s) in loading state_dict for MCC_GNN:
	Missing key(s) in state_dict: "ECC1.bias", "ECC1.nn.0.weight", "ECC1.nn.0.bias", "ECC1.nn.2.weight", "ECC1.nn.2.bias", "ECC1.lin.weight", "ECC2.bias", "ECC2.nn.0.weight", "ECC2.nn.0.bias", "ECC2.nn.2.weight", "ECC2.nn.2.bias", "ECC2.lin.weight". 
	Unexpected key(s) in state_dict: "conv1.att_src", "conv1.att_dst", "conv1.bias", "conv1.lin.weight", "conv2.att_src", "conv2.att_dst", "conv2.bias", "conv2.lin.weight". 

In [57]:
model = MCC_GNN(edge_dimension=1, node_dimension= 4, hidden_dimension= 64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
criterion = nn.SmoothL1Loss()


radmodel, optimizer, start_epoch = load_checkpoint(model, optimizer, "gnn_radiation_fully.pth")

Resumed training from epoch 99 (Previous Loss: 0.0001)


In [205]:
model, optimizer, start_epoch = load_checkpoint(model, optimizer,
                                                "gnn_gravity_fully.pth")

Resumed training from epoch 92 (Previous Loss: 0.0039)


load emperical data

In [63]:

from torch_geometric.loader import DataLoader
georgia = torch.load('.../data generation/datasets/version1/georgia2.pt', weights_only=False)
london = torch.load('.../data generation/datasets/version1/london.pt', weights_only=False)
japan = torch.load('.../data generation/datasets/version1/japan_data.pt', weights_only=False)


#covid time line
# g2019 = torch.load("/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/california_all_2019.pt", weights_only= False)
#
# g2020 = torch.load("/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/california_all_covid.pt", weights_only= False)
#
# g2021 = torch.load("/Users/emmapinckers/PycharmProjects/thesis/data generation/datasets/version1/california_all_2021.pt", weights_only= False)


complete_dataset = [japan, georgia, japan]


real_loader = DataLoader(complete_dataset, batch_size=1, shuffle=True) #batch size one to look at each graph individually




#evaluate model on real data

In [64]:
from torchmetrics import R2Score

radmodel.eval()

total_test_error = 0.0

#ignore location
FEATURE_TO_IGNORE = [2,3]
r2_metric = R2Score()

with torch.no_grad():
    for batch_data in real_loader:

        #the way i saved the data, location was accidentely saved too, but adding this to the model signficantly degraes perfromace so the features are set to 0
        batch_data.x[:, FEATURE_TO_IGNORE] = 0.0

        #get predictions
        predictions = model(batch_data)

        #error
        loss = criterion(predictions.view_as(batch_data.y), batch_data.y)
        total_test_error += loss.item()
        r2_metric.update(predictions.view_as(batch_data.y), batch_data.y)

avg_test_mse = total_test_error / len(real_loader)
final_r2 = r2_metric.compute()

print(f"Final Test: {avg_test_mse:.6f}")
print(f"final r2 {final_r2:.6f}")

r2_metric.reset()

Final Test (ignoring feature [2, 3]): 0.000177
final r2 0.961471


Covid prediciotns

In [75]:
import torch
from torch_geometric.loader import DataLoader
from torchmetrics import R2Score

complete_dataset = g2019 + g2020 + g2021 #add all year into one dataset

#shuffle=False so the order matches
real_loader = DataLoader(complete_dataset, batch_size=32, shuffle=False)

FEATURE_TO_IGNORE = [2, 3]
model.eval()

#track data
current_graph_index = 0
r2_metric = R2Score()

#make prediciotns
with torch.no_grad():
    for batch_data in real_loader:


        batch_data.x[:, FEATURE_TO_IGNORE] = 0.0
        predictions = model(batch_data)
        r2_metric.update(predictions.view_as(batch_data.y), batch_data.y)


        for i in range(len(predictions)):
            original_graph = complete_dataset[current_graph_index]

            original_graph.y_prediction = predictions[i].item()

            current_graph_index += 1
final_r2 = r2_metric.compute()
# print(f"final r2 {final_r2:.6f}")

r2_metric.reset()


#save predictions
torch.save(complete_dataset, "../data generation/datasets/version1/...pt")

NameError: name 'g2019' is not defined

In [65]:
batch_data.y

tensor([0.8027])

In [66]:
predictions

tensor([[0.8045]])